<div style="background-color:#B5121B;color:#ffffff;padding:16px 22px;border-radius:6px;font-family:Arial, Helvetica, sans-serif;">
<div style="font-size:12px;letter-spacing:0.08em;text-transform:uppercase;opacity:0.85;">PHYS7101 &middot; Numerical Methods and Data Analysis</div>
<div style="font-size:21px;font-weight:700;margin-top:6px;line-height:1.3;">Week 2 — Root-Finding: Solving Kepler's Equation</div>
</div>

#### Lecture 3 introduced bisection and Newton–Raphson on Kepler's equation. Here you implement both, find where each one wins and how Newton fails, and use your root-finder to compute an orbit.
***

## Where this fits

In the Week 1 extension you computed an orbit by stepping through time with Euler and RK4. Lecture 3 showed a second route. **Kepler's equation** gives the position on an elliptical orbit at any time directly:

$$M = E - e\sin E,$$

where $M = 2\pi t/T$ is the *mean anomaly* (known from the clock), $e$ is the eccentricity and $E$ is the *eccentric anomaly*, which gives the position: $x = a(\cos E - e)$, $y = b\sin E$.

There is no closed-form solution for $E$, so we solve $f(E) = E - e\sin E - M = 0$ numerically. Today you will:

1. implement bisection and Newton–Raphson;
2. compare how fast they converge, and reproduce the table from the lecture;
3. find out how Newton can go wrong;
4. check your answers against SciPy;
5. use your root-finder to compute an orbit, and compare it with RK4.

The functions you write take another function as an input (`bisection(f, a, b)`). If `lambda` or passing functions around is new to you, work through `week2_python_functions.ipynb` first (about 15 minutes).

**Nothing here is assessed.** Write your predictions in your logbook before you run each comparison.

## <span style="color:#B5121B">0. The problem (provided)</span>

$f(E) = E - e\sin E - M$ and its derivative $f'(E) = 1 - e\cos E$. Since $0 \le e < 1$, $f'(E) > 0$: $f$ is increasing, so there is exactly one root. Because $|E - M| = e|\sin E| \le e$, the root lies in $[M - e, M + e]$, a ready-made bracket.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def kepler_f(E, e, M):
    """Kepler's equation written as f(E) = 0."""
    return E - e * np.sin(E) - M

def kepler_fprime(E, e):
    """Derivative of kepler_f with respect to E."""
    return 1 - e * np.cos(E)

E = np.linspace(0, 2 * np.pi, 400)
M = 1.0
fig, ax = plt.subplots(figsize=(7, 4))
for e in [0.5, 0.9]:
    ax.plot(E, kepler_f(E, e, M), label=f"e = {e}")
ax.axhline(0, color="grey", lw=0.8)
ax.set_xlabel("Eccentric anomaly $E$ (rad)")
ax.set_ylabel("$f(E)$")
ax.set_title("Kepler's equation for $M = 1$")
ax.legend()
plt.show()

## <span style="color:#B5121B">1. Implement bisection</span>

From the lecture: if $f(a)\,f(b) < 0$, a root lies in $[a, b]$.

1. Take the midpoint $m = (a + b)/2$.
2. Keep the half in which the sign changes.
3. Repeat until the bracket is narrower than `tol`.

Write `bisection(f, a, b, tol)` returning the root and a list of the midpoints (the *history*), so we can count iterations later. Raise a `ValueError` if `f(a)` and `f(b)` have the same sign.

In [ ]:
def bisection(f, a, b, tol=1e-12, max_iter=200):
    """Find a root of f in [a, b] by bisection.

    Returns (root, history), where history lists the midpoints tried.
    """
    # TODO: your code here (delete the raise line when you start)
    raise NotImplementedError

<details><summary><b>Stuck? Open a hint</b> (try for 10 minutes first)</summary>

* Check the signs of `f(a)` and `f(b)` first; `raise ValueError(...)` if they match.
* Inside a `for` loop over `range(max_iter)`: compute the midpoint, append it to the history, and decide which end of the bracket it replaces. The product of two function values tells you whether they have opposite signs.
* `break` out of the loop once `b - a < tol`.

</details>

In [ ]:
# Test on e = 0.5, M = 1, bracket [M - e, M + e]
e, M = 0.5, 1.0
try:
    root, hist = bisection(lambda E: kepler_f(E, e, M), M - e, M + e)
    print(f"root E = {root:.12f} after {len(hist)} steps")
    print(f"f(root) = {kepler_f(root, e, M):.1e}")
    print(f"expected steps ≈ log2((b - a)/tol) = {np.log2(2 * e / 1e-12):.1f}")
except NotImplementedError:
    print("Not implemented yet: complete the function above, then re-run this cell.")

## <span style="color:#B5121B">2. Implement Newton–Raphson</span>

From the lecture: replace $f$ by its tangent at $x_n$ and move to where the tangent crosses zero,

$$x_{n+1} = x_n - \frac{f(x_n)}{f'(x_n)}.$$

Write `newton(f, fprime, x0, tol, max_iter)` returning the root and the history of iterates. Stop when the step $|x_{n+1} - x_n|$ is smaller than `tol`. If `max_iter` is reached first, print a warning: a root-finder that silently returns a wrong answer is worse than one that complains.

In [ ]:
def newton(f, fprime, x0, tol=1e-12, max_iter=50):
    """Find a root of f by Newton-Raphson, starting from x0.

    Returns (root, history), where history lists every iterate (x0 included).
    """
    # TODO: your code here (delete the raise line when you start)
    raise NotImplementedError

<details><summary><b>Stuck? Open a hint</b> (try for 10 minutes first)</summary>

* Keep the current estimate and a list starting with `x0`.
* Each pass of the loop: compute the step from the formula above, update the estimate, append it, and `return` as soon as the step is smaller than `tol`.
* If the loop ends without returning, print a warning, then return what you have.

</details>

In [ ]:
e, M = 0.5, 1.0
try:
    root, hist = newton(lambda E: kepler_f(E, e, M), lambda E: kepler_fprime(E, e), x0=M)
    print(f"root E = {root:.12f} after {len(hist) - 1} steps")
except NotImplementedError:
    print("Not implemented yet: complete the function above, then re-run this cell.")

## <span style="color:#B5121B">3. Round 1: how fast?</span>

**Before running, predict in your logbook:** how many iterations will each method need to reach $10^{-12}$?

For $M = 1$ and $e = 0.5$ and $0.9$:

1. find an accurate reference root (`scipy.optimize.brentq` with `xtol=1e-15` is fine);
2. run both methods, starting Newton from $E_0 = M$ and bisection on $[M - e, M + e]$;
3. plot the error $|E_n - E_\text{exact}|$ against iteration $n$ on a log scale, for both methods and both eccentricities;
4. print Newton's errors and compare them with the table on Lecture 3, slide 7.

In [ ]:
from scipy.optimize import brentq

# TODO: for e in [0.5, 0.9]:
#     E_exact = brentq(...)
#     run bisection and newton, convert each history to errors abs(E_n - E_exact)
#     plot errors against iteration number (ax.semilogy)
#     print Newton's errors

In your logbook: how many decimal digits does each method gain per step? Does the eccentricity change bisection's step count? Why not?

## <span style="color:#B5121B">4. Round 2: break it</span>

The lecture listed ways Newton can fail. Try two.

**(a) High eccentricity.** Take $e = 0.99$ and $M = 0.2$. Start Newton from $E_0 = M$, then from $E_0 = 0$, then from $E_0 = \pi$. For each, print the number of iterations and the largest $|E|$ visited (`max(abs(np.array(history)))`). What happens to Newton's first step, and why? (Look at $f'(E)$ near $E = 0$.) Then try `max_iter=10` from $E_0 = M$.

**(b) Cycling.** Newton on $f(x) = x^3 - 2x + 2$, starting from $x_0 = 0$. Print the first few iterates. Then find the root with bisection on $[-3, 0]$.

In [ ]:
# (a) high eccentricity
e, M = 0.99, 0.2
# TODO: for E0 in [M, 0.0, np.pi]: run newton, print iterations and max |E| visited
# TODO: repeat E0 = M with max_iter=10

# (b) cycling
g = lambda x: x**3 - 2 * x + 2
gp = lambda x: 3 * x**2 - 2
# TODO: run newton(g, gp, 0.0, max_iter=10) and print the history
# TODO: find the root with bisection on [-3, 0]

## <span style="color:#B5121B">5. Check against SciPy</span>

In practice you'd use a library routine. The lecture recommended `scipy.optimize.brentq` (bracketed, guaranteed and fast). `scipy.optimize.newton` is also available. Solve $e = 0.9$, $M = 1$ with both and check that they agree with your functions to $10^{-12}$. Use `full_output=True` to see how many iterations they needed.

In [ ]:
from scipy.optimize import newton as sp_newton

e, M = 0.9, 1.0
# TODO: root_brent, info = brentq(..., full_output=True)
# TODO: root_sp = sp_newton(..., fprime=..., x0=...)
# TODO: compare with your bisection and newton results

## <span style="color:#B5121B">6. From Kepler's equation to an orbit</span>

The Week 1 extension suggested an eccentric orbit: $GM = 1$, starting at $(x, y) = (1, 0)$ with velocity $(0, 0.7)$. The cell below works out its orbital elements (semi-major axis $a$, eccentricity $e$, period $T$) and integrates one period with RK4, as in Week 1.

The body starts at the far point of the orbit (apoapsis), which corresponds to $E = \pi$. So here $M = \pi + 2\pi t/T$, and, with the orbit's long axis along $x$,

$$x = -a(\cos E - e), \qquad y = -b\sin E, \qquad b = a\sqrt{1 - e^2}.$$

In [ ]:
GM = 1.0
x0, y0, vx0, vy0 = 1.0, 0.0, 0.0, 0.7

# Orbital elements from the initial conditions (energy and angular momentum)
r0 = np.hypot(x0, y0)
a = 1 / (2 / r0 - (vx0**2 + vy0**2) / GM)        # semi-major axis
h_ang = x0 * vy0 - y0 * vx0                       # angular momentum per unit mass
ecc = np.sqrt(1 - h_ang**2 / (GM * a))            # eccentricity
b = a * np.sqrt(1 - ecc**2)                       # semi-minor axis
T = 2 * np.pi * np.sqrt(a**3 / GM)                # period
print(f"a = {a:.4f}, e = {ecc:.4f}, b = {b:.4f}, T = {T:.4f}")

def orbit_deriv(s):
    x, y, vx, vy = s
    r3 = (x**2 + y**2) ** 1.5
    return np.array([vx, vy, -GM * x / r3, -GM * y / r3])

def rk4_orbit(state0, h, n_steps):
    s = np.array(state0, dtype=float)
    out = [s.copy()]
    for _ in range(n_steps):
        k1 = orbit_deriv(s)
        k2 = orbit_deriv(s + h / 2 * k1)
        k3 = orbit_deriv(s + h / 2 * k2)
        k4 = orbit_deriv(s + h * k3)
        s = s + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
        out.append(s.copy())
    return np.array(out)

h_step = 0.01
n_steps = int(round(T / h_step))
t_rk4 = np.arange(n_steps + 1) * h_step
orbit_rk4 = rk4_orbit((x0, y0, vx0, vy0), h_step, n_steps)

For each time in `t_rk4`, solve Kepler's equation for $E$ with your `newton` function, convert to $(x, y)$, and:

1. plot both orbits on the same axes (`ax.set_aspect("equal")`);
2. print the largest distance between the two positions at the same time.

Which method would you trust to give the position after 1000 orbits, and why?

<details><summary><b>Stuck? Open a hint</b> (try for 10 minutes first)</summary>

Loop over `t_rk4`. For each time, compute $M$, then fix $e$ and $M$ with a `lambda` so that `newton` sees a function of $E$ only (`week2_python_functions.ipynb`, Section 4). `M` itself is a good starting value. Store $x$ and $y$ in arrays, then compare with `orbit_rk4[:, 0]` and `orbit_rk4[:, 1]`.

</details>

In [ ]:
# TODO: Kepler positions at every time in t_rk4; plot both orbits; largest distance between them

## The verdict

In your logbook:

1. When would you use bisection rather than Newton–Raphson, and why does Brent's method exist?
2. What starting guess would you use for Kepler's equation at any eccentricity?
3. When is solving an equation for the state at time $t$ better than stepping through time, and when is it impossible?

## <span style="color:#FD9029">Extension A (optional): minima are roots of the gradient</span>

From the lecture: at a smooth minimum $f'(x) = 0$, so optimisation is root-finding applied to the gradient. Take $f(x) = x^4 - 3x^2 + x$.

1. Plot $f$ on $[-2, 2]$.
2. **Gradient descent**: $x_{n+1} = x_n - \eta f'(x_n)$ with $\eta = 0.01$ and 2000 steps, from $x_0 = -0.5$ and from $x_0 = +0.5$. Which minimum does each find?
3. Use your `newton` on $f'(x)$ (with $f''(x)$ as the derivative) from the same two starts. What has it found from $x_0 = +0.5$? Check the sign of $f''$ there.
4. Compare with `scipy.optimize.minimize`.

In [ ]:
from scipy.optimize import minimize

f_opt = lambda x: x**4 - 3 * x**2 + x
fp_opt = lambda x: 4 * x**3 - 6 * x + 1
fpp_opt = lambda x: 12 * x**2 - 6
# TODO

## <span style="color:#FD9029">Extension B (optional): Newton in two dimensions</span>

The lecture's example: find where the circle $x^2 + y^2 = 4$ meets the curve $y = e^x$. Write $\mathbf{F}(x, y) = (x^2 + y^2 - 4,\; e^x - y) = \mathbf{0}$ with Jacobian

$$J = \begin{pmatrix} 2x & 2y \\ e^x & -1 \end{pmatrix}.$$

Each Newton step solves $J\,\Delta\mathbf{x} = -\mathbf{F}$ (use `np.linalg.solve`; never invert $J$). Find both intersections by trying different starting points, including $(0, 0)$. Check with `scipy.optimize.root`.

In [ ]:
from scipy.optimize import root

F = lambda v: np.array([v[0]**2 + v[1]**2 - 4, np.exp(v[0]) - v[1]])
J = lambda v: np.array([[2 * v[0], 2 * v[1]], [np.exp(v[0]), -1.0]])
# TODO

<hr style="border:none;border-top:1px solid #BEC0C2;margin-top:32px;">
<div style="color:#555656;font-size:12px;margin-top:8px;font-family:Arial, Helvetica, sans-serif;">
PHYS7101 &mdash; Numerical Methods and Data Analysis
</div>